In [0]:
# ============================================
# CAMADA SILVER - Limpeza, Qualidade e LGPD
# ============================================
from pyspark.sql.functions import (
    current_timestamp, lit, col, sha2, concat_ws,
    upper, trim, to_timestamp, when, count
)

BUCKET_PATH = "s3://data-lake-portfolio-priscila-2026"
BRONZE_PATH = f"{BUCKET_PATH}/bronze"
SILVER_PATH = f"{BUCKET_PATH}/silver"

# ── 1. CLIENTES ──────────────────────────────
df_clientes = spark.read.format("delta").load(f"{BRONZE_PATH}/clientes")

# Qualidade: checar nulos
print("🔍 Checagem de nulos - Clientes:")
df_clientes.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_clientes.columns
]).show()

df_clientes_silver = df_clientes \
    .dropDuplicates(["cliente_id"]) \
    .filter(col("cliente_id").isNotNull()) \
    .withColumn("nome",   upper(trim(col("nome")))) \
    .withColumn("cidade", upper(trim(col("cidade")))) \
    .withColumn("cpf_anonimizado",   sha2(col("cpf"), 256)) \
    .withColumn("email_anonimizado", sha2(col("email"), 256)) \
    .drop("cpf", "email") \
    .withColumn("_silver_at", current_timestamp())

df_clientes_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/clientes")
print(f"✅ Silver/clientes: {df_clientes_silver.count()} linhas | CPF e email anonimizados (LGPD)\n")

# ── 2. TRANSAÇÕES ────────────────────────────
df_transacoes = spark.read.format("delta").load(f"{BRONZE_PATH}/transacoes")

df_transacoes_silver = df_transacoes \
    .dropDuplicates(["transacao_id"]) \
    .filter(col("valor") > 0) \
    .filter(col("status").isin("Aprovada", "Negada", "Em Analise")) \
    .withColumn("data_hora", to_timestamp(col("data_hora"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("_silver_at", current_timestamp())

df_transacoes_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/transacoes")
print(f"✅ Silver/transacoes: {df_transacoes_silver.count()} linhas\n")

# ── 3. LOGS DE SEGURANÇA ─────────────────────
df_logs = spark.read.format("delta").load(f"{BRONZE_PATH}/logs_seguranca")

df_logs_silver = df_logs \
    .dropDuplicates(["log_id"]) \
    .filter(col("score_risco_dispositivo").between(0, 100)) \
    .withColumn("data_hora_login", to_timestamp(col("data_hora_login"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("ip_origem", sha2(col("ip_origem"), 256)) \
    .withColumn("_silver_at", current_timestamp())

df_logs_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/logs_seguranca")
print(f"✅ Silver/logs_seguranca: {df_logs_silver.count()} linhas | IPs anonimizados (LGPD)\n")

print("🥈 Camada Silver concluída!")